# Evaluation Metric: Average Domain RMSE

This notebook explains the evaluation metric used for the competition.

---

## Prediction and Ground Truth Format

Predictions are compared against ground-truth PM2.5 concentration fields with the following shape:

(N, H, W, T)

where:
- N is the number of evaluation samples  
- H × W is the spatial grid resolution (140 × 124)  
- T is the forecast horizon (16 hours)

Predictions must exactly match the ground-truth shape and contain only finite values.

---

## Public and Private Leaderboards

The evaluation dataset is internally split into public and private subsets. The same metric is applied to both splits. Participants receive feedback only on the public leaderboard during the competition, while the private leaderboard determines the final rankings.

---

## Metric Definition

### Step 1: Spatial RMSE per Forecast Hour

For each sample and each forecast hour, RMSE is computed over the full spatial domain(H,W):

$$
\mathrm{RMSE}_{n,t}
= \sqrt{
\frac{1}{HW}
\sum_{i=1}^{H} \sum_{j=1}^{W}
\left(
\hat{y}_{n,i,j,t} - y_{n,i,j,t}
\right)^2
}
$$

This results in an array of shape:`(N, T)`

---

### Step 2: Averaging Over Time and Samples

The final score is obtained by averaging RMSE values across:
- all forecast hours (T = 16)
- all evaluation samples (N)

Final Score = $$
\mathrm{mean}_{n,t} RMSE(n, t)$$

Lower scores indicate better spatio-temporal predictive performance.

---

In [ ]:
## Reference Implementation

import os
import numpy as np
import pandas as pd

# -----------------------
# Participant visible errors
# -----------------------

class ParticipantVisibleError(Exception):
    """
    Raising errors for participants for faulty submissions
    """
    pass



# -----------------------
# Constructing a submission.csv
# -----------------------

def generate_submission_csv(submission_dir):
    """
    Constructs a minimal submission.csv pointing to preds.npy.
    """
    pred_path = os.path.join(submission_dir, "preds.npy")

    df = pd.DataFrame({
        "id": ["evaluation"],
        "path": [pred_path]
    })

    df.to_csv("submission.csv", index=False)

# -----------------------
# Metric implementation
# -----------------------


def average_domain_rmse(y_true, y_pred):
    rmse_vals = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=(1, 2)))
    return float(np.mean(rmse_vals))


def score(solution_df, submission_df):
    sol_path = solution_df["path"].iloc[0]
    pred_path = submission_df["path"].iloc[0]

    if not os.path.exists(pred_path):
        raise ParticipantVisibleError("Prediction file not found")

    y_pred = np.load(pred_path).astype(np.float32)
    y_true = np.load(sol_path).astype(np.float32)

    if y_pred.shape != y_true.shape:
        raise ParticipantVisibleError("Prediction array has incorrect shape")

    if not np.isfinite(y_pred).all():
        raise ParticipantVisibleError("Prediction array contains NaN or Inf values")

    score_val = average_domain_rmse(y_true, y_pred)

    return score_val
